In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

pio.renderers.default = "vscode"
print("✅ Imports done")

✅ Imports done


In [2]:
ROOT     = Path(r"C:\Users\sayan\Desktop\Data_Analysis\ai_sustainability_project")
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "charts"
MDL_DIR  = ROOT / "outputs" / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MDL_DIR.mkdir(parents=True, exist_ok=True)

FILE1 = DATA_DIR / "AI_DataCenter_Sustainability_Datasets.xlsx"
FILE2 = DATA_DIR / "AI_Boom_vs_DC_Sustainability_FINAL.xlsx"

PALETTE = {
    'cyan':    '#00B4D8',
    'blue':    '#1D4ED8',
    'green':   '#10B981',
    'red':     '#EF4444',
    'orange':  '#F97316',
    'purple':  '#8B5CF6',
    'yellow':  '#F59E0B',
    'grey':    '#8B949E',
}

DARK_TEMPLATE = {
    'paper_bgcolor': '#0D1117',
    'plot_bgcolor':  '#161B22',
    'font':          dict(color='#E6EDF3', family='Arial'),
    'legend':        dict(bgcolor='#161B22', bordercolor='#30363D'),
}

def make_fig(**kwargs):
    fig = go.Figure(**kwargs)
    fig.update_layout(**DARK_TEMPLATE)
    return fig

print("✅ Config ready")

✅ Config ready


In [3]:
def load_sheet(filepath, sheet_index, header_row=3):
    df = pd.read_excel(filepath, sheet_name=sheet_index, header=header_row)
    df.columns = (df.columns.astype(str)
                  .str.replace(r'\n', ' ', regex=True)
                  .str.strip()
                  .str.replace(r'\s+', ' ', regex=True))
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    return df

def drop_note_rows(df, col_index=0):
    col = df.columns[col_index]
    return df[~df[col].astype(str).str.startswith('📝')].reset_index(drop=True)

def force_numeric(df, exclude=None):
    exclude = exclude or []
    for col in df.columns:
        if col not in exclude:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Load the two sheets we need for regression
df_core_ts = drop_note_rows(force_numeric(
    load_sheet(FILE2, 2),
    exclude=['Data Flag (H=Historical P=Projected)']
))

df_ai_kpis = drop_note_rows(force_numeric(
    load_sheet(FILE2, 4),
    exclude=['Data Flag (H/P)']
))

df_scenarios = drop_note_rows(force_numeric(
    load_sheet(FILE2, 9),
    exclude=[]
))

# Split historical vs projected
flag_col     = [c for c in df_core_ts.columns if 'Flag' in c][0]
kpis_flag    = [c for c in df_ai_kpis.columns if 'Flag' in c or 'flag' in c][0]

df_hist      = df_core_ts[df_core_ts[flag_col].str.strip() == 'H'].copy()
df_proj      = df_core_ts[df_core_ts[flag_col].str.strip() == 'P'].copy()
df_kpis_hist = df_ai_kpis[df_ai_kpis[kpis_flag].str.strip() == 'H'].copy()

# Merge training compute index into historical data
compute_col = [c for c in df_kpis_hist.columns if 'Compute' in c and 'Index' in c][0]
df_hist['Year']      = df_hist['Year'].astype(int)
df_kpis_hist['Year'] = df_kpis_hist['Year'].astype(int)

df_hist = df_hist.merge(
    df_kpis_hist[['Year', compute_col]],
    on='Year', how='left'
)

print(f"✅ Data loaded")
print(f"   Historical rows : {len(df_hist)}")
print(f"   Columns         : {list(df_hist.columns)}")

✅ Data loaded
   Historical rows : 10
   Columns         : ['Year', 'Global DC Energy (TWh)', 'AI Share of DC Energy (%)', 'Hyperscaler Electricity (TWh)', 'Global AI Private Inv. ($B)', 'GenAI Investment ($B)', 'GPU/AI Chip Revenue ($B)', 'GPU Shipments for AI (M units)', 'Hyperscale DC Count', 'DC Carbon Emissions (MtCO2e)', 'Renewable % of DC Electricity', 'Global DC Capex ($B)', 'Data Flag (H=Historical P=Projected)', 'Training Compute Index (2017=1)']


In [4]:
# ── What we are doing ────────────────────────────────────────────────────────
# Based on the correlation rankings from Step 2, we select our features
# carefully. We exclude Hyperscaler TWh and DC Carbon because they are
# too directly derived from our target variable (DC Energy TWh) — including
# them would be data leakage, making the model look better than it really is.
#
# TARGET:   Global DC Energy (TWh)
# FEATURES: Hyperscale Count, AI Energy Share %, GenAI Investment,
#           GPU Revenue, GPU Shipments, Renewable %, Training Compute Index,
#           AI Investment, DC Capex

TARGET = 'Global DC Energy (TWh)'

FEATURES = [
    'Hyperscale DC Count',
    'AI Share of DC Energy (%)',
    'GenAI Investment ($B)',
    'GPU/AI Chip Revenue ($B)',
    'GPU Shipments for AI (M units)',
    'Renewable % of DC Electricity',
    compute_col,
    'Global AI Private Inv. ($B)',
    'Global DC Capex ($B)',
]

# Keep only rows where target and at least 4 features are non-null
df_model = df_hist[['Year', TARGET] + FEATURES].copy()
df_model = df_model.dropna(subset=[TARGET])
df_model = df_model[df_model[FEATURES].notna().sum(axis=1) >= 4]

# Fill remaining missing feature values with column median
# We use median not mean because some features have skewed distributions
for col in FEATURES:
    if df_model[col].isna().any():
        median_val = df_model[col].median()
        df_model[col] = df_model[col].fillna(median_val)
        print(f"   Filled missing values in '{col}' with median: {median_val:.2f}")

print(f"\n✅ Feature matrix ready")
print(f"   Rows available for training : {len(df_model)}")
print(f"   Features                    : {len(FEATURES)}")
print(f"\n   Feature list:")
for i, f in enumerate(FEATURES, 1):
    print(f"   {i:2}. {f}")

   Filled missing values in 'GPU/AI Chip Revenue ($B)' with median: 16.50
   Filled missing values in 'GPU Shipments for AI (M units)' with median: 1.10
   Filled missing values in 'Training Compute Index (2017=1)' with median: 23.50
   Filled missing values in 'Global AI Private Inv. ($B)' with median: 47.40

✅ Feature matrix ready
   Rows available for training : 10
   Features                    : 9

   Feature list:
    1. Hyperscale DC Count
    2. AI Share of DC Energy (%)
    3. GenAI Investment ($B)
    4. GPU/AI Chip Revenue ($B)
    5. GPU Shipments for AI (M units)
    6. Renewable % of DC Electricity
    7. Training Compute Index (2017=1)
    8. Global AI Private Inv. ($B)
    9. Global DC Capex ($B)


In [5]:
# ── Why we split this way ────────────────────────────────────────────────────
# This is a time series — each year depends on the previous years.
# A random split would let the model train on 2023 data and test on 2019,
# which is cheating — you can't use future data to predict the past.
# We always split chronologically: train on earlier years, test on later ones.
#
# Split: 2015–2021 for training (7 years), 2022–2024 for testing (3 years)
# This tests whether the model trained on pre-AI-boom data can predict
# the post-boom acceleration — a genuinely hard and meaningful test.

df_model = df_model.sort_values('Year').reset_index(drop=True)

TRAIN_END = 2021
TEST_START = 2022

train = df_model[df_model['Year'] <= TRAIN_END].copy()
test  = df_model[df_model['Year'] >= TEST_START].copy()

X_train = train[FEATURES].values
y_train = train[TARGET].values
X_test  = test[FEATURES].values
y_test  = test[TARGET].values

# Scale features — zero mean, unit variance
# Required for Linear and Ridge regression
# XGBoost doesn't need scaling but it doesn't hurt
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Train / test split complete")
print(f"   Training set : {len(train)} years "
      f"({int(train['Year'].min())}–{int(train['Year'].max())})")
print(f"   Test set     : {len(test)} years  "
      f"({int(test['Year'].min())}–{int(test['Year'].max())})")
print(f"   X_train shape: {X_train_scaled.shape}")
print(f"   X_test shape : {X_test_scaled.shape}")
print(f"\n   Training target range: "
      f"{y_train.min():.0f}–{y_train.max():.0f} TWh")
print(f"   Test target range    : "
      f"{y_test.min():.0f}–{y_test.max():.0f} TWh")

✅ Train / test split complete
   Training set : 7 years (2015–2021)
   Test set     : 3 years  (2022–2024)
   X_train shape: (7, 9)
   X_test shape : (3, 9)

   Training target range: 194–265 TWh
   Test target range    : 310–415 TWh


In [6]:
# ── What we are doing ────────────────────────────────────────────────────────
# We train four models on the same data and compare them.
# This is best practice — never rely on a single model.
#
# Linear Regression  : simplest baseline, fully interpretable coefficients
# Ridge Regression   : linear with regularisation — handles correlated features
#                      better than plain linear (our features ARE correlated)
# Random Forest      : ensemble of decision trees, captures non-linear patterns
# XGBoost            : gradient boosting, typically the best performer on
#                      small tabular datasets
#
# We evaluate each on the test set using three metrics:
#   RMSE  — root mean squared error (in TWh — same units as target)
#   MAE   — mean absolute error (average absolute miss in TWh)
#   R²    — proportion of variance explained (1.0 = perfect)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(
                             n_estimators=200,
                             max_depth=3,
                             min_samples_leaf=2,
                             random_state=42
                         ),
    'XGBoost':           xgb.XGBRegressor(
                             n_estimators=200,
                             max_depth=2,
                             learning_rate=0.05,
                             subsample=0.8,
                             colsample_bytree=0.8,
                             random_state=42,
                             verbosity=0
                         ),
}

results = {}

print("Training models...\n")
print(f"{'Model':<22} {'RMSE (TWh)':>12} {'MAE (TWh)':>12} {'R²':>8}")
print("-" * 58)

for name, model in models.items():
    # Use scaled features for linear models, unscaled for tree models
    if name in ['Linear Regression', 'Ridge Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results[name] = {
        'model':  model,
        'y_pred': y_pred,
        'rmse':   rmse,
        'mae':    mae,
        'r2':     r2,
    }

    print(f"{name:<22} {rmse:>12.2f} {mae:>12.2f} {r2:>8.3f}")

print("\n✅ All models trained")

Training models...

Model                    RMSE (TWh)    MAE (TWh)       R²
----------------------------------------------------------
Linear Regression             15.77        11.35    0.866
Ridge Regression              53.88        33.88   -0.569
Random Forest                133.64       126.55   -8.654
XGBoost                      109.65       100.86   -5.499

✅ All models trained


In [7]:
# ── Why cross validation matters here ────────────────────────────────────────
# With only 10 rows, a single train/test split can be lucky or unlucky.
# TimeSeriesSplit gives us multiple train/test windows while always
# respecting the chronological order — no future data leaks into training.
#
# We use 3 splits, which with 10 rows gives us:
#   Split 1: train on 4 years, test on 2 years
#   Split 2: train on 6 years, test on 2 years
#   Split 3: train on 8 years, test on 2 years
# The average RMSE across all splits is a more honest performance estimate.

tscv = TimeSeriesSplit(n_splits=3)

print("Cross-validation results (TimeSeriesSplit, 3 folds)\n")
print(f"{'Model':<22} {'CV RMSE Mean':>14} {'CV RMSE Std':>13}")
print("-" * 52)

for name, res in results.items():
    model = res['model']

    if name in ['Linear Regression', 'Ridge Regression']:
        X_all = scaler.fit_transform(df_model[FEATURES].values)
    else:
        X_all = df_model[FEATURES].values

    y_all = df_model[TARGET].values

    cv_scores = cross_val_score(
        model, X_all, y_all,
        cv=tscv,
        scoring='neg_root_mean_squared_error'
    )
    cv_rmse = -cv_scores

    results[name]['cv_rmse_mean'] = cv_rmse.mean()
    results[name]['cv_rmse_std']  = cv_rmse.std()

    print(f"{name:<22} {cv_rmse.mean():>14.2f} {cv_rmse.std():>13.2f}")

print("\n✅ Cross validation complete")

Cross-validation results (TimeSeriesSplit, 3 folds)

Model                    CV RMSE Mean   CV RMSE Std
----------------------------------------------------
Linear Regression               18.39          3.88
Ridge Regression                32.06          8.56
Random Forest                   79.51         39.06
XGBoost                         56.59         22.12

✅ Cross validation complete


In [8]:
# ── What we are doing ────────────────────────────────────────────────────────
# Select the best model based on CV RMSE — the most honest metric.
# Then extract and interpret its coefficients or feature importances.
# This is where the ML output becomes a business insight.

# Select best model by lowest CV RMSE
best_name = min(results, key=lambda k: results[k]['cv_rmse_mean'])
best      = results[best_name]

print("=" * 58)
print(f"  BEST MODEL: {best_name}")
print("=" * 58)
print(f"  Test RMSE : {best['rmse']:.2f} TWh")
print(f"  Test MAE  : {best['mae']:.2f} TWh")
print(f"  Test R²   : {best['r2']:.3f}")
print(f"  CV RMSE   : {best['cv_rmse_mean']:.2f} ± {best['cv_rmse_std']:.2f} TWh")

# Actual vs predicted on test set
print(f"\n  Year-by-year test set predictions:")
print(f"  {'Year':>6} {'Actual (TWh)':>14} {'Predicted (TWh)':>16} {'Error (TWh)':>12}")
print(f"  {'-'*52}")
for yr, actual, pred in zip(
    test['Year'].values, y_test, best['y_pred']
):
    error = pred - actual
    print(f"  {int(yr):>6} {actual:>14.1f} {pred:>16.1f} "
          f"{error:>+12.1f}")

# Feature importance or coefficients
print(f"\n  Feature Importance / Coefficients:")
print(f"  {'-'*52}")

if best_name in ['Linear Regression', 'Ridge Regression']:
    coefs = best['model'].coef_
    for feat, coef in sorted(
        zip(FEATURES, coefs), key=lambda x: abs(x[1]), reverse=True
    ):
        bar = '█' * int(abs(coef) / max(abs(coefs)) * 20)
        print(f"  {feat:<40} {coef:>+8.3f}  {bar}")

else:
    importances = best['model'].feature_importances_
    for feat, imp in sorted(
        zip(FEATURES, importances), key=lambda x: x[1], reverse=True
    ):
        bar = '█' * int(imp * 40)
        print(f"  {feat:<40} {imp:>8.3f}  {bar}")

# Save best model to disk
joblib.dump(best['model'], MDL_DIR / f"best_model_{best_name.replace(' ','_')}.pkl")
joblib.dump(scaler,        MDL_DIR / "feature_scaler.pkl")
print(f"\n✅ Best model saved → outputs/models/")

  BEST MODEL: Linear Regression
  Test RMSE : 15.77 TWh
  Test MAE  : 11.35 TWh
  Test R²   : 0.866
  CV RMSE   : 18.39 ± 3.88 TWh

  Year-by-year test set predictions:
    Year   Actual (TWh)  Predicted (TWh)  Error (TWh)
  ----------------------------------------------------
    2022          310.0            311.2         +1.2
    2023          370.0            343.5        -26.5
    2024          415.0            408.6         -6.4

  Feature Importance / Coefficients:
  ----------------------------------------------------
  Global DC Capex ($B)                      +33.652  ████████████████████
  AI Share of DC Energy (%)                 -20.706  ████████████
  Renewable % of DC Electricity             +13.929  ████████
  Hyperscale DC Count                        +9.731  █████
  Global AI Private Inv. ($B)                -9.114  █████
  GPU/AI Chip Revenue ($B)                   +6.018  ███
  Training Compute Index (2017=1)            -2.147  █
  GPU Shipments for AI (M units)   

In [17]:
# ── Fix: refit scaler only on training data, then transform consistently ─────

# Refit scaler strictly on training data only
scaler_fixed = StandardScaler()
scaler_fixed.fit(X_train)

X_train_fixed = scaler_fixed.transform(X_train)
X_test_fixed  = scaler_fixed.transform(X_test)
X_all_fixed   = scaler_fixed.transform(df_model[FEATURES].values)

# Retrain best model cleanly on fixed scaled data
from sklearn.linear_model import LinearRegression
best_model_fixed = LinearRegression()
best_model_fixed.fit(X_train_fixed, y_train)

# Predictions
y_train_pred_fixed = best_model_fixed.predict(X_train_fixed)
y_test_pred_fixed  = best_model_fixed.predict(X_test_fixed)
y_all_pred_fixed   = best_model_fixed.predict(X_all_fixed)

# Verify test predictions match Cell 8 exactly
print("Verification — test set predictions should match Cell 8:")
print(f"{'Year':>6} {'Actual':>10} {'Predicted':>12} {'Error':>10}")
print("-" * 42)
for yr, actual, pred in zip(test['Year'].values, y_test, y_test_pred_fixed):
    print(f"{int(yr):>6} {actual:>10.1f} {pred:>12.1f} {pred-actual:>+10.1f}")

years_all  = df_model['Year'].values.astype(int)
y_all_true = df_model[TARGET].values

fig = make_fig()

# Shaded train region
fig.add_vrect(
    x0=years_all.min() - 0.5, x1=TRAIN_END + 0.5,
    fillcolor=PALETTE['blue'], opacity=0.06,
    layer='below', line_width=0,
    annotation_text='Training period',
    annotation_position='top left',
    annotation_font=dict(color=PALETTE['blue'], size=10)
)

# Shaded test region
fig.add_vrect(
    x0=TEST_START - 0.5, x1=years_all.max() + 0.5,
    fillcolor=PALETTE['orange'], opacity=0.06,
    layer='below', line_width=0,
    annotation_text='Test period',
    annotation_position='top right',
    annotation_font=dict(color=PALETTE['orange'], size=10)
)

# Actual values
fig.add_trace(go.Scatter(
    x=years_all, y=y_all_true,
    name='Actual DC Energy (TWh)',
    mode='lines+markers',
    line=dict(color=PALETTE['cyan'], width=3),
    marker=dict(size=9),
    hovertemplate='Year: %{x}<br>Actual: %{y:.1f} TWh<extra></extra>'
))

# Predicted values — full period
fig.add_trace(go.Scatter(
    x=years_all, y=y_all_pred_fixed,
    name='Model Prediction (Linear Regression)',
    mode='lines+markers',
    line=dict(color=PALETTE['yellow'], width=2.5, dash='dash'),
    marker=dict(size=8, symbol='diamond'),
    hovertemplate='Year: %{x}<br>Predicted: %{y:.1f} TWh<extra></extra>'
))

# Error bars on test years only
for yr, actual, pred in zip(
    test['Year'].values.astype(int), y_test, y_test_pred_fixed
):
    fig.add_shape(
        type='line',
        x0=yr, x1=yr,
        y0=min(actual, pred), y1=max(actual, pred),
        line=dict(
            color=PALETTE['red'] if abs(actual - pred) > 20
                  else PALETTE['green'],
            width=2, dash='dot'
        )
    )
    fig.add_annotation(
        x=yr, y=(actual + pred) / 2,
        text=f'{pred - actual:+.1f} TWh',
        showarrow=False,
        font=dict(
            size=10,
            color=PALETTE['red'] if abs(actual - pred) > 20
                  else PALETTE['green']
        ),
        xshift=35
    )

# 2023 structural break annotation
fig.add_annotation(
    x=2023, y=y_test_pred_fixed[1],
    text='2023: Model underestimates<br>by 26.5 TWh — ChatGPT<br>structural break',
    showarrow=True, arrowhead=2,
    arrowcolor=PALETTE['red'],
    ax=-120, ay=60,
    font=dict(size=10, color=PALETTE['red']),
    bgcolor='#21262D',
    bordercolor=PALETTE['red'], borderpad=5
)

fig.update_layout(
    title=dict(
        text='Linear Regression: Predicted vs Actual DC Energy (2015–2024)',
        font=dict(size=16, color='#E6EDF3'), x=0.5
    ),
    xaxis=dict(title='Year', dtick=1, gridcolor='#21262D'),
    yaxis=dict(
        title='DC Electricity Consumption (TWh)',
        gridcolor='#21262D'
    ),
    height=540,
    legend=dict(x=0.02, y=0.98),
    annotations=list(fig.layout.annotations) + [
        go.layout.Annotation(
            text=(f'Test R² = {best["r2"]:.3f}  |  '
                  f'RMSE = {best["rmse"]:.1f} TWh  |  '
                  f'CV RMSE = {best["cv_rmse_mean"]:.1f} ± '
                  f'{best["cv_rmse_std"]:.1f} TWh'),
            x=0.5, y=-0.12, xref='paper', yref='paper',
            showarrow=False,
            font=dict(size=10, color='#8B949E')
        )
    ]
)

# Save fixed scaler and model
joblib.dump(best_model_fixed, MDL_DIR / "best_model_Linear_Regression.pkl")
joblib.dump(scaler_fixed,     MDL_DIR / "feature_scaler.pkl")

fig.write_html(str(OUT_DIR / "03_regression_predictions.html"))
fig.show()
print("\n✅ Chart saved with corrected predictions")

Verification — test set predictions should match Cell 8:
  Year     Actual    Predicted      Error
------------------------------------------
  2022      310.0        311.2       +1.2
  2023      370.0        343.5      -26.5
  2024      415.0        408.6       -6.4



✅ Chart saved with corrected predictions


In [18]:
# ── What this chart shows ─────────────────────────────────────────────────────
# The regression coefficients visualised as a horizontal bar chart.
# This is the chart that translates the ML output into business language.
# Each bar represents: a one-unit increase in this feature is associated
# with X TWh change in DC energy demand, holding all else constant.

coefs   = best_model.coef_
feat_df = pd.DataFrame({
    'Feature':     FEATURES,
    'Coefficient': coefs
}).sort_values('Coefficient', key=abs, ascending=True)

colors = [
    PALETTE['green'] if c > 0 else PALETTE['red']
    for c in feat_df['Coefficient']
]

fig = make_fig()

fig.add_trace(go.Bar(
    x=feat_df['Coefficient'],
    y=feat_df['Feature'],
    orientation='h',
    marker=dict(color=colors, line=dict(color='#21262D', width=0.5)),
    text=[f'{v:+.2f}' for v in feat_df['Coefficient']],
    textposition='outside',
    textfont=dict(size=10, color='#E6EDF3'),
    hovertemplate='<b>%{y}</b><br>Coefficient: %{x:+.3f} TWh<extra></extra>',
))

fig.add_vline(x=0, line_color='#8B949E', line_width=1.5)

# Callout for top feature
fig.add_annotation(
    x=33.6, y='Global DC Capex ($B)',
    text='+33.6 TWh per $1B capex',
    showarrow=True, arrowhead=2,
    arrowcolor=PALETTE['yellow'],
    ax=60, ay=-30,
    font=dict(size=10, color=PALETTE['yellow']),
    bgcolor='#21262D',
    bordercolor=PALETTE['yellow'], borderpad=4
)

fig.update_layout(
    title=dict(
        text='Regression Coefficients: What Drives DC Energy Demand?',
        font=dict(size=16, color='#E6EDF3'), x=0.5
    ),
    xaxis=dict(
        title=dict(
            text='Coefficient (TWh change per unit increase in feature)',
            standoff=25        # pushes axis title down away from tick labels
        ),
        gridcolor='#21262D',
        zeroline=True, zerolinecolor='#8B949E'
    ),
    yaxis=dict(title='', gridcolor='#21262D'),
    height=520,
    showlegend=False,
    margin=dict(l=250, r=120, b=100),   # extra bottom margin clears the overlap
    annotations=list(fig.layout.annotations) + [
        go.layout.Annotation(
            text='Green = positive driver of DC energy  |  '
                 'Red = negative coefficient (see interpretation notes)',
            x=0.5, y=-0.18,            # pushed further down
            xref='paper', yref='paper',
            showarrow=False,
            font=dict(size=9, color='#8B949E')
        )
    ]
)

fig.write_html(str(OUT_DIR / "03_feature_importance.html"))
fig.show()
print("✅ Chart saved → outputs/charts/03_feature_importance.html")

✅ Chart saved → outputs/charts/03_feature_importance.html


In [19]:
print("=" * 62)
print("  STEP 3 — REGRESSION FINDINGS SUMMARY")
print("=" * 62)

print(f"""
MODEL PERFORMANCE
  Best model    : Linear Regression
  Test R²       : {best['r2']:.3f}
  Test RMSE     : {best['rmse']:.1f} TWh
  CV RMSE       : {best['cv_rmse_mean']:.1f} ± {best['cv_rmse_std']:.1f} TWh
  Avg error %   : {(best['mae'] / y_test.mean() * 100):.1f}% of mean target value

KEY BUSINESS INSIGHTS FROM COEFFICIENTS
  → DC Capex is the #1 driver: every $1B in DC investment
    is associated with +{coefs[FEATURES.index('Global DC Capex ($B)')]:.1f} TWh additional
    annual energy demand

  → Each new hyperscale DC added globally is associated
    with +{coefs[FEATURES.index('Hyperscale DC Count')]:.1f} TWh additional annual demand
    At 130 new DCs per year → +{coefs[FEATURES.index('Hyperscale DC Count')] * 130:.0f} TWh/yr
    additional energy pressure

  → Renewable % coefficient is positive (+{coefs[FEATURES.index('Renewable % of DC Electricity')]:.1f})
    confirming Chart 5: renewables grow WITH energy
    consumption, not instead of it

THE 2023 STRUCTURAL BREAK
  → Model underestimated 2023 by 26.5 TWh
  → This is quantitative evidence that ChatGPT created
    a genuine structural break in the AI-energy relationship
  → Pre-boom models systematically underforecast
    post-boom energy demand

SAVED FILES
  → outputs/models/best_model_Linear_Regression.pkl
  → outputs/models/feature_scaler.pkl
  → outputs/charts/03_regression_predictions.html
  → outputs/charts/03_feature_importance.html

Next → 04_forecasting.ipynb
""")

  STEP 3 — REGRESSION FINDINGS SUMMARY

MODEL PERFORMANCE
  Best model    : Linear Regression
  Test R²       : 0.866
  Test RMSE     : 15.8 TWh
  CV RMSE       : 18.4 ± 3.9 TWh
  Avg error %   : 3.1% of mean target value

KEY BUSINESS INSIGHTS FROM COEFFICIENTS
  → DC Capex is the #1 driver: every $1B in DC investment
    is associated with +33.7 TWh additional
    annual energy demand

  → Each new hyperscale DC added globally is associated
    with +9.7 TWh additional annual demand
    At 130 new DCs per year → +1265 TWh/yr
    additional energy pressure

  → Renewable % coefficient is positive (+13.9)
    confirming Chart 5: renewables grow WITH energy
    consumption, not instead of it

THE 2023 STRUCTURAL BREAK
  → Model underestimated 2023 by 26.5 TWh
  → This is quantitative evidence that ChatGPT created
    a genuine structural break in the AI-energy relationship
  → Pre-boom models systematically underforecast
    post-boom energy demand

SAVED FILES
  → outputs/models/best_m